# Essay Writer Walkthrough

This notebook rewrites the essay-writer workflow into small, inline steps.
Each stage is broken into its own cell so you can see exactly how state changes over time.

## Workflow Map

The original graph-based version had these logical steps:

1. Plan the essay
2. Decide what to research
3. Collect research notes
4. Generate a draft
5. Critique the draft
6. Research the critique
7. Generate a revised draft

Below, each step is written inline instead of being hidden inside node functions.

In [1]:
# Import standard library helpers used to read environment variables.
import os

# Import typing helpers so the shared state structure is explicit.
from typing import List, TypedDict

# Import the message classes used to build chat-style prompts.
from langchain_core.messages import HumanMessage, SystemMessage

# Import the chat model wrapper so we can call the LLM.
from langchain_openai import ChatOpenAI

# Import a simple search tool so the notebook can gather outside information.
from langchain_community.tools import DuckDuckGoSearchRun

# No extra parser library is required here because we will parse plain text query lists.


/Users/zhenqiyang/Documents/github/llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define the shape of the data that moves through the essay workflow.
class AgentState(TypedDict):
    # Store the original assignment or essay request.
    task: str
    # Store the outline created during the planning step.
    plan: str
    # Store the latest essay draft.
    draft: str
    # Store feedback about the current draft.
    critique: str
    # Store all research notes gathered so far.
    content: List[str]
    # Track how many generations have happened.
    revision_number: int
    # Define the maximum number of generations to allow.
    max_revisions: int

# Create the initial shared state for a sample run.
state: AgentState = {
    # Choose the topic we want the workflow to write about.
    "task": "What is the difference between LangChain and LangSmith?",
    # Start with an empty outline because planning has not run yet.
    "plan": "",
    # Start with no draft because generation has not run yet.
    "draft": "",
    # Start with no critique because nothing has been reviewed yet.
    "critique": "",
    # Start with an empty research list because no search has happened yet.
    "content": [],
    # Set the current generation count to 1 for the first draft pass.
    "revision_number": 1,
    # Limit the workflow so it does not revise forever.
    "max_revisions": 2,
}

# Print the starting state so it is easy to inspect before any model calls.
state


{'task': 'What is the difference between LangChain and LangSmith?',
 'plan': '',
 'draft': '',
 'critique': '',
 'content': [],
 'revision_number': 1,
 'max_revisions': 2}

In [3]:
# Read the DeepSeek API key from the environment.
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY", "").strip()

# Read the DeepSeek base URL and remove a trailing slash if one exists.
deepseek_base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").strip().rstrip("/")

# Read the model name and default to DeepSeek chat if none is provided.
deepseek_model = os.getenv("DEEPSEEK_MODEL", "deepseek-chat").strip()

# Fail early if the API key is missing so the problem is obvious.
if not deepseek_api_key:
    raise RuntimeError("DEEPSEEK_API_KEY is missing from your environment or .env file.")

# Normalize the base URL so it ends with /v1 because the OpenAI-compatible client expects that form.
if not deepseek_base_url.endswith("/v1"):
    deepseek_base_url = f"{deepseek_base_url}/v1"

# Create the chat model object that will handle planning, writing, and critique.
model = ChatOpenAI(
    # Tell the wrapper which remote model to call.
    model=deepseek_model,
    # Provide the API key used for authentication.
    api_key=deepseek_api_key,
    # Point the wrapper to the DeepSeek OpenAI-compatible endpoint.
    base_url=deepseek_base_url,
)

# Create the search tool that will fetch outside information.
search_tool = DuckDuckGoSearchRun()

# Show the normalized connection settings so they can be checked quickly.
{
    "model": deepseek_model,
    "base_url": deepseek_base_url,
}


{'model': 'deepseek-chat', 'base_url': 'https://api.deepseek.com/v1'}

In [4]:
# Tell the model how to create a high-level essay outline.
PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay.
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes
or instructions for the sections."""

# Tell the model how to write the essay using the outline and research notes.
WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.
Generate the best essay possible for the user's request and the initial outline.
If the user provides critique, respond with a revised version of your previous attempts.
Utilize all the information below as needed:

------

{content}"""

# Tell the model how to critique the current essay draft.
REFLECTION_PROMPT = """You are a teacher grading an essay submission.
Generate critique and recommendations for the user's submission.
Provide detailed recommendations, including requests for length, depth, style, etc."""

# Tell the model how to create research queries before writing the first draft.
RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can
be used when writing the following essay. Generate a list of search queries that will gather
any relevant information. Only generate 3 queries max."""

# Tell the model how to create research queries after critique arrives.
RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can
be used when making any requested revisions (as outlined below).
Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""


In [5]:
# Define a small helper that turns a plain-text LLM response into a Python list of queries.
def parse_queries(raw_text: str) -> List[str]:
    # Split the response into individual lines.
    lines = raw_text.splitlines()
    # Start an empty list that will hold clean query strings.
    queries = []
    # Loop through each line and normalize it.
    for line in lines:
        # Remove surrounding whitespace first.
        cleaned = line.strip()
        # Remove common numbered-list prefixes like `1. ` or `- `.
        cleaned = cleaned.lstrip("-*").strip()
        # If the line starts with a digit, remove the leading numbering token.
        if cleaned and cleaned[0].isdigit() and "." in cleaned:
            cleaned = cleaned.split(".", 1)[1].strip()
        # Keep only non-empty lines after cleanup.
        if cleaned:
            queries.append(cleaned)
    # Return at most three queries to match the prompt contract.
    return queries[:3]

# Quick sanity check for the helper.
parse_queries("1. query one\n2. query two\n- query three")


['query one', 'query two', 'query three']

## Step 1: Plan The Essay

This is the logic that used to live inside `plan_node`.

In [6]:
# Build the chat messages for the planning step.
plan_messages = [
    # Use the system prompt to define the model's role as an outline writer.
    SystemMessage(content=PLAN_PROMPT),
    # Pass the user's essay topic as the human request.
    HumanMessage(content=state["task"]),
]

# Call the model to generate the outline.
plan_response = model.invoke(plan_messages)

# Save the outline back into shared state.
state["plan"] = plan_response.content

# Show the outline so we can inspect the planning result.
state["plan"]


'# Outline: LangChain vs. LangSmith – Understanding Their Roles in AI Application Development\n\n## I. Introduction\n- **Hook**: The rapid growth of LLM-powered applications has led to a proliferation of development tools.\n- **Context**: Introduce LangChain and LangSmith as key components in the modern LLM development stack.\n- **Thesis Statement**: While both are products of LangChain Inc., LangChain is an open-source framework for building LLM applications, whereas LangSmith is a commercial platform for debugging, testing, and monitoring those applications in production.\n- **Roadmap**: This essay will examine their distinct purposes, architectures, use cases, and how they complement each other in the development lifecycle.\n\n## II. Understanding LangChain: The Development Framework\n### A. Core Purpose and Philosophy\n- Open-source Python/JavaScript library for orchestrating LLM workflows\n- Designed to simplify the integration of LLMs with external data sources and tools\n- Modul

## Step 2: Ask The Model What To Research

This is the logic that used to live inside `research_plan_node` before the search loop.

In [7]:
# Ask the model to propose a few useful search queries for the essay topic.
plan_query_response = model.invoke([
    # Use the research-planning system instruction.
    SystemMessage(content=RESEARCH_PLAN_PROMPT + "\nReturn only a short list of search queries, one per line, with no explanation."),
    # Use the essay task as the thing to research.
    HumanMessage(content=state["task"]),
])

# Parse the plain-text model response into a normal Python list.
plan_queries = parse_queries(plan_query_response.content)

# Show the exact search queries before running them.
plan_queries


['LangChain vs LangSmith comparison',
 'LangChain LangSmith features',
 'LangSmith use cases examples']

## Step 3: Collect Research Notes

This cell makes the search loop explicit instead of hiding it in a helper function.

In [8]:
# Start a fresh list to hold text snippets collected from search.
research_notes = []

# Loop over each query suggested by the model.
for query in plan_queries:
    # Print the current query so we know what is being searched.
    print(f"Searching for: {query}")
    # Run the search tool and get a text result back.
    search_result = search_tool.invoke(query)
    # Keep the raw result so the writer can use it later.
    research_notes.append(search_result)

# Store the research notes inside shared state.
state["content"] = research_notes

# Show how many research entries were collected.
len(state["content"])


Searching for: LangChain vs LangSmith comparison
Searching for: LangChain LangSmith features
Searching for: LangSmith use cases examples


3

In [9]:
# Join the research notes into a readable preview string.
research_preview = "\n\n---\n\n".join(state["content"])

# Print the preview so the gathered context is visible before generation.
print(research_preview[:4000])


Discover the key differences betweenLangChainandLangSmithin this in-depth guide. Compare features, integration options, and benefits. CompleteLangChainecosystemcomparison:LangChainfor building chains, LangGraph for complex agents,LangSmithfor monitoring. Decision matrix, code examples, and use cases for each tool. CompareLangChain, LangGraph,LangSmith, and LangFlow. Learn their roles, strengths, and when to use each for building production-ready AI applications. CompareLangChain, LangGraph,LangSmith, and Langfuse to understand which AI development tools you actually need for your LLM projects. Learn the key differences betweenLangChain, LangGraph, andLangSmith. Discover how each tool fits into the LLM application stack and when to use them.

---

LangSmithworks with any LLM framework. Trace applications built with OpenAI SDK, Anthropic SDK, Vercel AI SDK, LlamaIndex, or custom implementations, not justLangChain. Featuresand Functionality. Here’s whereLangchaingets interesting. Its main

## Step 4: Generate The First Draft

This is the inline version of what used to live inside `generation_node`.

In [10]:
# Combine the research notes into one long context string for the writer prompt.
combined_content = "\n\n".join(state["content"])

# Build the user-facing message that includes both the task and the plan.
writer_human_message = HumanMessage(
    # Tell the writer model what to write and include the outline it should follow.
    content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}"
)

# Build the full message list for the draft generation step.
writer_messages = [
    # Inject the research notes into the writer system prompt.
    SystemMessage(content=WRITER_PROMPT.format(content=combined_content)),
    # Provide the task plus outline as the user request.
    writer_human_message,
]

# Ask the model to write the first draft.
draft_response = model.invoke(writer_messages)

# Save the generated essay draft into shared state.
state["draft"] = draft_response.content

# Increment the generation counter because one full draft has now been produced.
state["revision_number"] = state["revision_number"] + 1

# Show the current draft.
state["draft"]


"# LangChain vs. LangSmith – Understanding Their Roles in AI Application Development\n\nThe explosive growth of large language model (LLM) applications has spurred a parallel expansion in development tools, creating a complex ecosystem for builders to navigate. Central to this landscape are LangChain and LangSmith, two prominent offerings from LangChain Inc. that are often mentioned together yet serve distinctly different purposes. While LangChain is an open-source framework for constructing LLM applications, LangSmith is a commercial platform for debugging, testing, and monitoring those applications in production. This essay will examine their unique roles, architectures, and primary use cases, demonstrating that they are not competitors but complementary tools designed for different stages of the AI development lifecycle.\n\nLangChain functions as the essential development framework, a modular toolkit for orchestrating LLM workflows. Its core philosophy is to simplify the integration

## Step 5: Critique The Draft

This is the inline version of what used to live inside `reflection_node`.

In [11]:
# Build the messages for the reflection step.
critique_messages = [
    # Tell the model to act like a teacher giving feedback.
    SystemMessage(content=REFLECTION_PROMPT),
    # Send the current draft as the thing to critique.
    HumanMessage(content=state["draft"]),
]

# Ask the model to review the draft.
critique_response = model.invoke(critique_messages)

# Save the critique back into shared state.
state["critique"] = critique_response.content

# Show the critique so the revision goals are explicit.
state["critique"]


'# Critique and Recommendations for "LangChain vs. LangSmith – Understanding Their Roles in AI Application Development"\n\n## Overall Assessment\nThis is a well-structured, clear, and informative essay that effectively distinguishes between LangChain and LangSmith. The writing is professional, the comparisons are insightful, and the conclusion ties everything together nicely. The essay successfully achieves its stated purpose of clarifying these tools\' complementary roles. However, there are opportunities to enhance depth, add practical examples, and strengthen certain sections.\n\n## Strengths\n- **Clear thesis and structure**: The introduction establishes the purpose effectively, and each paragraph builds logically toward the conclusion.\n- **Accurate technical descriptions**: Your explanations of Chains, Agents, Memory, Tracing, and Evaluation demonstrate solid understanding.\n- **Effective comparison framework**: The dimensions of comparison (nature/accessibility, primary function

## Step 6: Research The Critique

This is the inline version of what used to live inside `research_critique_node`.

In [12]:
# Ask the model to turn the critique into additional search queries.
critique_query_response = model.invoke([
    # Use the prompt that focuses on research for revision.
    SystemMessage(content=RESEARCH_CRITIQUE_PROMPT + "\nReturn only a short list of search queries, one per line, with no explanation."),
    # Feed in the critique so the model knows what needs improvement.
    HumanMessage(content=state["critique"]),
])

# Parse the plain-text model response into a normal Python list.
critique_queries = parse_queries(critique_query_response.content)

# Show the new queries that will support the revision pass.
critique_queries


['LangSmith pricing model vs open source alternatives',
 'LangChain abstraction complexity criticisms',
 'LLM application development tooling evolution trends']

In [13]:
# Copy the existing research notes so we keep the original context.
revised_research_notes = list(state["content"])

# Loop over each critique-driven query.
for query in critique_queries:
    # Print the current query so the new research path is visible.
    print(f"Searching for critique follow-up: {query}")
    # Run the search tool for the critique-driven topic.
    search_result = search_tool.invoke(query)
    # Append the new material to the growing context list.
    revised_research_notes.append(search_result)

# Save the expanded research context back into shared state.
state["content"] = revised_research_notes

# Show the total number of research entries after the revision search pass.
len(state["content"])


Searching for critique follow-up: LangSmith pricing model vs open source alternatives
Searching for critique follow-up: LangChain abstraction complexity criticisms
Searching for critique follow-up: LLM application development tooling evolution trends


6

## Step 7: Generate The Revised Draft

This is the same generation logic as before, but now it uses the critique-informed research context.

In [14]:
# Merge all current research notes into one larger context string.
revised_combined_content = "\n\n".join(state["content"])

# Build a new user message that includes the task, the plan, and the critique.
revision_human_message = HumanMessage(
    # Ask for a better draft using both the original plan and the critique guidance.
    content=(
        f"{state['task']}\n\n"
        f"Here is my plan:\n\n{state['plan']}\n\n"
        f"Please improve the draft using this critique:\n\n{state['critique']}"
    )
)

# Build the messages for the revision generation step.
revision_messages = [
    # Provide the expanded research context to the writer.
    SystemMessage(content=WRITER_PROMPT.format(content=revised_combined_content)),
    # Provide the user request with explicit revision instructions.
    revision_human_message,
]

# Ask the model to produce the revised essay.
revision_response = model.invoke(revision_messages)

# Overwrite the draft with the improved version.
state["draft"] = revision_response.content

# Increment the generation counter again because another draft was created.
state["revision_number"] = state["revision_number"] + 1

# Show the revised essay draft.
state["draft"]


'# LangChain vs. LangSmith – Understanding Their Roles in AI Application Development\n\nAs LLM applications evolve from experimental prototypes to mission-critical systems, the tooling ecosystem has matured in parallel, offering specialized solutions for different stages of the development lifecycle. Within this landscape, LangChain and LangSmith have emerged as prominent but frequently confused tools. While both originate from LangChain Inc., they serve fundamentally different purposes: LangChain is an open-source framework for building LLM applications, whereas LangSmith is a commercial platform for debugging, testing, and monitoring those applications in production. This essay will examine their distinct architectures, primary use cases, and how they complement each other to form a comprehensive development ecosystem.\n\nLangChain serves as the foundational development framework—essentially a modular toolkit for orchestrating LLM workflows. Think of it as the equivalent of Django or

## Step 8: Decide Whether To Continue

This reproduces the old conditional logic in plain notebook form.

In [15]:
# Compare the current generation count to the allowed maximum.
should_stop = state["revision_number"] > state["max_revisions"]

# Print the decision in a human-readable way.
if should_stop:
    # Explain that the workflow would end here in the original graph.
    print("Stop here: the workflow has exceeded the configured max_revisions.")
else:
    # Explain that another critique-and-rewrite cycle would happen next.
    print("Continue: the workflow would go back to critique and revision.")

# Show the final state so every field can be inspected together.
state


Stop here: the workflow has exceeded the configured max_revisions.


{'task': 'What is the difference between LangChain and LangSmith?',
 'plan': '# Outline: LangChain vs. LangSmith – Understanding Their Roles in AI Application Development\n\n## I. Introduction\n- **Hook**: The rapid growth of LLM-powered applications has led to a proliferation of development tools.\n- **Context**: Introduce LangChain and LangSmith as key components in the modern LLM development stack.\n- **Thesis Statement**: While both are products of LangChain Inc., LangChain is an open-source framework for building LLM applications, whereas LangSmith is a commercial platform for debugging, testing, and monitoring those applications in production.\n- **Roadmap**: This essay will examine their distinct purposes, architectures, use cases, and how they complement each other in the development lifecycle.\n\n## II. Understanding LangChain: The Development Framework\n### A. Core Purpose and Philosophy\n- Open-source Python/JavaScript library for orchestrating LLM workflows\n- Designed to s

## Mapping Back To The Original Graph

These inline sections correspond to the original node functions:

- `plan_node` -> Step 1
- `research_plan_node` -> Steps 2 and 3
- `generation_node` -> Steps 4 and 7
- `reflection_node` -> Step 5
- `research_critique_node` -> Step 6
- `should_continue` -> Step 8

Once the logic is clear, you can turn each section back into a LangGraph node if you want the orchestration benefits again.

## Rebuild The Workflow As LangGraph Nodes

Now that the step-by-step logic is clear, the next section wraps the same workflow back into LangGraph nodes.

In [16]:
# Import the graph builder and the END sentinel used by LangGraph.
from langgraph.graph import END, StateGraph

# Import an in-memory checkpointer so repeated runs can keep thread state.
from langgraph.checkpoint.memory import MemorySaver

# Create a memory object that LangGraph can use while the notebook kernel is alive.
memory = MemorySaver()


In [17]:
# Define the node that creates the essay outline.
def plan_node(state: AgentState):
    # Build the messages for the planning step.
    messages = [
        # Tell the model to act like an expert essay planner.
        SystemMessage(content=PLAN_PROMPT),
        # Pass in the essay topic from the current state.
        HumanMessage(content=state["task"]),
    ]
    # Call the model to generate the outline.
    response = model.invoke(messages)
    # Return only the field this node updates.
    return {"plan": response.content}

# Define the node that generates research queries for the initial draft.
def research_plan_node(state: AgentState):
    # Ask for short search queries in plain text so the code works with DeepSeek.
    response = model.invoke([
        # Use the planning research prompt plus a formatting instruction.
        SystemMessage(content=RESEARCH_PLAN_PROMPT + "\nReturn only a short list of search queries, one per line, with no explanation."),
        # Pass the essay topic as the research target.
        HumanMessage(content=state["task"]),
    ])
    # Turn the raw model output into a Python list of query strings.
    queries = parse_queries(response.content)
    # Start from any existing notes already in state.
    content = list(state.get("content", []))
    # Search each generated query and keep the text results.
    for query in queries:
        # Call the search tool once for the current query.
        result = search_tool.invoke(query)
        # Save the returned text into the shared content list.
        content.append(result)
    # Return the updated research notes.
    return {"content": content}

# Define the node that writes or rewrites the essay draft.
def generation_node(state: AgentState):
    # Merge all research notes into one context block.
    combined_content = "\n\n".join(state.get("content", []))
    # Start the human prompt with the task and the outline.
    user_text = f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}"
    # If critique exists, include it so the model knows how to improve the draft.
    if state.get("critique"):
        user_text += f"\n\nPlease improve the draft using this critique:\n\n{state['critique']}"
    # Build the generation messages.
    messages = [
        # Inject the gathered research into the writer prompt.
        SystemMessage(content=WRITER_PROMPT.format(content=combined_content)),
        # Pass the task, plan, and optional critique to the model.
        HumanMessage(content=user_text),
    ]
    # Generate the current draft.
    response = model.invoke(messages)
    # Return the new draft and increment the draft counter.
    return {
        "draft": response.content,
        "revision_number": state.get("revision_number", 1) + 1,
    }

# Define the node that critiques the current draft.
def reflection_node(state: AgentState):
    # Build the reflection messages.
    messages = [
        # Tell the model to act like a teacher and reviewer.
        SystemMessage(content=REFLECTION_PROMPT),
        # Send the latest draft for evaluation.
        HumanMessage(content=state["draft"]),
    ]
    # Request the critique.
    response = model.invoke(messages)
    # Return the critique text.
    return {"critique": response.content}

# Define the node that researches the critique before the next revision.
def research_critique_node(state: AgentState):
    # Ask for search queries derived from the critique text.
    response = model.invoke([
        # Use the critique research prompt plus a strict output format instruction.
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT + "\nReturn only a short list of search queries, one per line, with no explanation."),
        # Pass the critique so the model knows what to research next.
        HumanMessage(content=state["critique"]),
    ])
    # Parse the returned plain-text queries.
    queries = parse_queries(response.content)
    # Copy existing notes so the earlier research is preserved.
    content = list(state.get("content", []))
    # Search each critique-driven query and append the results.
    for query in queries:
        # Fetch more outside information for the revision step.
        result = search_tool.invoke(query)
        # Add the new result text to the cumulative content list.
        content.append(result)
    # Return the expanded research context.
    return {"content": content}

# Define the branching rule that decides whether the graph stops or loops.
def should_continue(state: AgentState):
    # Stop once the graph has produced more drafts than the configured maximum.
    if state["revision_number"] > state["max_revisions"]:
        return END
    # Otherwise continue into the critique branch.
    return "reflect"


In [18]:
# Create a new state graph whose state shape matches AgentState.
builder = StateGraph(AgentState)

# Register each workflow step as a named graph node.
builder.add_node("planner", plan_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_critique", research_critique_node)

# Declare the first node the graph should run.
builder.set_entry_point("planner")

# Wire the initial writing path.
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")

# Wire the revision loop path.
builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")

# Add the conditional branch that either stops or goes back for critique.
builder.add_conditional_edges(
    # Evaluate the state after each generation step.
    "generate",
    # Use the routing function defined above.
    should_continue,
    # Map each routing output to a graph destination.
    {END: END, "reflect": "reflect"},
)

# Compile the builder into an executable LangGraph app.
graph = builder.compile(checkpointer=memory)

# Return the compiled graph object so it can be inspected.
graph


ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [19]:
# Create a thread id so LangGraph can associate checkpoints with one run.
thread = {"configurable": {"thread_id": "essay-writer-demo"}}

# Define the initial graph input.
graph_input = {
    # Provide the essay topic.
    "task": state["task"],
    # Set empty placeholders for fields that the graph will fill in.
    "plan": "",
    "draft": "",
    "critique": "",
    "content": [],
    # Start the generation counter at 1 to match the earlier walkthrough.
    "revision_number": 1,
    # Limit the workflow to the same maximum revision count.
    "max_revisions": 2,
}

# Stream the graph execution so each node update is visible.
for event in graph.stream(graph_input, thread):
    # Print each node result as it arrives.
    print(event)


{'planner': {'plan': '# Outline: LangChain vs. LangSmith – Understanding Their Roles in LLM Development\n\n## I. Introduction\n- **Hook**: The rapid evolution of LLM frameworks and tools has created specialized platforms for different stages of development.\n- **Background**: Brief overview of the LangChain ecosystem as a response to LLM integration challenges.\n- **Thesis Statement**: While LangChain and LangSmith are complementary products within the same ecosystem, they serve fundamentally different purposes—LangChain as a development framework for building LLM applications, and LangSmith as an observability and debugging platform for monitoring and improving those applications.\n- **Roadmap**: This essay will define each tool, analyze their core functions, compare their roles in the development lifecycle, and explore how they work together.\n\n## II. Understanding LangChain: The Development Framework\n- **Definition**: Open-source framework for building applications powered by lang

## Visualize The Graph

The first visualization path tries to render an image. If `pygraphviz` is unavailable, the fallback still prints a readable graph representation.

In [23]:
# Import notebook display helpers for images and rich output.
from IPython.display import Image, Markdown, display

# Get the internal drawable graph representation.
graph_view = graph.get_graph()

# # Always print an ASCII version so there is at least one visualization path that stays local.
# print(graph_view.draw_ascii())

# Try to render a PNG image if the optional drawing dependency is installed.
try:
    # Display a PNG version of the graph when pygraphviz is available.
    display(Image(graph_view.draw_png()))
except Exception as exc:
    # Explain why the PNG path failed instead of hiding the error.
    print(f"PNG rendering was not available: {exc}")
    # Fall back to Mermaid source so the structure is still inspectable.
    mermaid_diagram = graph_view.draw_mermaid()
    # Show the Mermaid diagram text in a code block.
    display(Markdown(f"```mermaid\n{mermaid_diagram}\n```"))


PNG rendering was not available: Install pygraphviz to draw graphs: `pip install pygraphviz`.


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	research_plan(research_plan)
	generate(generate)
	reflect(reflect)
	research_critique(research_critique)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	generate -.-> __end__;
	generate -.-> reflect;
	planner --> research_plan;
	reflect --> research_critique;
	research_critique --> generate;
	research_plan --> generate;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```